# Limpeza dos Dados - E-commerce

Nesta etapa, a base será tratada sem remover pedidos válidos indevidamente.

Critérios usados:
- manter os pedidos quando o `pedido_id` for único;
- preencher valores ausentes em colunas importantes;
- corrigir o `valor_total` usando `quantidade * valor_unitario`;
- corrigir datas em formatos diferentes;
- não remover o pedido inteiro quando a data de entrega estiver inválida.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('data/ecommerce_pedidos.csv')
df.head()

## 1. Verificando informações iniciais

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 2. Verificando duplicatas

Aqui o mais seguro é verificar duplicidade pelo `pedido_id`, pois ele representa o identificador único do pedido.

In [ ]:
df['pedido_id'].duplicated().sum()

In [ ]:
df = df.drop_duplicates(subset='pedido_id')
df.shape

## 3. Tratando valores ausentes

Como são poucos casos, vamos preencher com `Não informado` para não perder pedidos que ainda podem ser úteis na análise.

In [ ]:
df['cliente'] = df['cliente'].fillna('Não informado')
df['estado'] = df['estado'].fillna('Não informado')
df['pagamento'] = df['pagamento'].fillna('Não informado')

df.isnull().sum()

## 4. Corrigindo o valor total

O `valor_total` será recalculado usando `quantidade * valor_unitario`. Assim, se algum valor original estava errado, negativo ou inconsistente, ele será corrigido.

In [ ]:
df['valor_total_original'] = df['valor_total']
df['valor_total'] = df['quantidade'] * df['valor_unitario']
df[['quantidade', 'valor_unitario', 'valor_total_original', 'valor_total']].head()

## 5. Tratando datas

A base possui datas em formatos diferentes. Por isso, usamos `dayfirst=True` para ajudar o pandas a interpretar datas no formato brasileiro também.

In [ ]:
# Tratando data_pedido
df["data_pedido"] = pd.to_datetime(df["data_pedido"], errors="coerce")

# Tratando data_entrega
df["data_entrega"] = pd.to_datetime(df["data_entrega"], errors="coerce")

# Calculando dias de entrega
df["dias_entrega"] = (df["data_entrega"] - df["data_pedido"]).dt.days

# Corrigindo entregas inválidas
datas_invalidas = df["dias_entrega"] < 0

df.loc[datas_invalidas, "data_entrega"] = pd.NaT
df.loc[datas_invalidas, "dias_entrega"] = None

## 6. Tratando datas de entrega inválidas

Se a data de entrega estiver antes da data do pedido, não vamos excluir o pedido inteiro. Vamos apenas deixar a entrega como vazia, pois o erro está na data de entrega.

In [ ]:
datas_invalidas = df['data_entrega'] < df['data_pedido']
datas_invalidas.sum()

In [ ]:
df.loc[datas_invalidas, 'data_entrega'] = pd.NaT
df['dias_entrega'] = (df['data_entrega'] - df['data_pedido']).dt.days

df[['data_pedido', 'data_entrega', 'dias_entrega']].head()

## 7. Criando colunas auxiliares

In [ ]:
df['mes'] = df['data_pedido'].dt.month
df['ano'] = df['data_pedido'].dt.year

# Pedido válido para faturamento líquido: tudo que não foi cancelado
df['pedido_valido'] = df['status'] != 'Cancelado'

df.head()

## 8. Salvando a base limpa

In [ ]:
df.to_csv('data/ecommerce_pedidos_limpo.csv', index=False)
df.shape